issue:
* there mismatch where gender in customer table is not same with the customer table from erp, hence we decide to make the gender is crm soruce is main source, fill unknown row with the value from erp cutsomer table if exists

In [0]:
%sql
SELECT 
ROW_NUMBER()OVER (ORDER BY a.customer_id) AS customer_key,
a.customer_id,
a.customer_number,
a.first_name,
a.last_name,
CASE WHEN a.GENDER!="Unknown" THEN a.gender
ELSE coalesce(b.gender,"Unknown")
END AS gender,
a.marital_status,
b.birth_date,
c.country,
a.create_date
from workspace.silver.crm_cust_info AS a
LEFT JOIN workspace.silver.erp_cust_az12 AS b
ON a.customer_number = b.customer_id
LEFT JOIN workspace.silver.erp_loc_a101 AS c
ON a.customer_number = c.customer_id



In [0]:
df = spark.sql("""
               SELECT 
ROW_NUMBER()OVER (ORDER BY a.customer_id) AS customer_key,
a.customer_id,
a.customer_number,
a.first_name,
a.last_name,
CASE WHEN a.GENDER!="Unknown" THEN a.gender
ELSE coalesce(b.gender,"Unknown")
END AS gender,
a.marital_status,
b.birth_date,
c.country,
a.create_date
from workspace.silver.crm_cust_info AS a
LEFT JOIN workspace.silver.erp_cust_az12 AS b
ON a.customer_number = b.customer_id
LEFT JOIN workspace.silver.erp_loc_a101 AS c
ON a.customer_number = c.customer_id      
               """)

df.display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")